# DeepGuard AI - Master Google Colab Training Notebook
### End-to-End Automated Deepfake Model Training (Zero Manual Local Upload)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliKhan-Devs/Deep-Learning/blob/main/07_DeepGuard_AI/training/notebooks/00_quickstart_colab_training.ipynb)

**Everything you need in ONE notebook:**
1. **Clones / updates repository** directly from GitHub (no manual folder uploading from your local PC).
2. **Downloads deepfake dataset directly via URL** straight to Colab NVMe (no manual download to your PC and upload to Google Drive).
3. **Trains either EfficientNet-B0 or XceptionNet** with Mixed Precision (AMP), AdamW, and Early Stopping.
4. **Evaluates model** (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix).
5. **Auto-downloads `best_model.pth`** directly to your browser for immediate use in the local DeepGuard AI web app!

---
## Step 1: Environment Setup & Clone / Update Repository
Verify GPU acceleration and ensure latest repository code is loaded.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

# 2. Clone or pull repository to ensure latest code
import os
import sys

%cd /content
if os.path.exists('/content/Deep-Learning'):
    print("Updating existing repository clone...")
    !cd /content/Deep-Learning && git pull origin main
else:
    print("Cloning repository from GitHub...")
    !git clone https://github.com/AliKhan-Devs/Deep-Learning.git

%cd /content/Deep-Learning/07_DeepGuard_AI/training

# 3. Install required packages
!pip install -q timm opencv-python-headless scikit-learn matplotlib

# 4. Add src to Python path
SRC_PATH = "/content/Deep-Learning/07_DeepGuard_AI/training/src"
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f"\n>>> Environment ready in: {os.getcwd()}")
print(f">>> Python sys.path includes: {SRC_PATH}")

---
## Step 2: Automated Direct URL Dataset Download
Downloads real and manipulated face crops directly into Colab's NVMe storage.

- **Hugging Face DeepFakeFace (`OpenRL/DeepFakeFace`)**: Direct URL download with real human faces and InsightFace deepfake swaps. **No account or login required.**
- Fully self-contained execution.

In [ ]:
#@title Dataset Configuration & Direct URL Ingestion { run: "auto" }
DATASET_SOURCE = "HuggingFace_OpenRL_DeepFakeFace" #@param ["HuggingFace_OpenRL_DeepFakeFace", "Kaggle_140k_Faces", "Existing_Local_Path"]
MAX_SAMPLES_PER_CLASS = 3000 #@param {type:"integer"} # Set to e.g. 3000 for fast training (10-15 mins), or 0 for all
DATA_DIR = "/content/data"

import os
import sys
import glob
import urllib.request
import zipfile
import pandas as pd
import numpy as np

# Self-contained robust downloader functions
def download_file_with_progress(url, output_path):
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    print(f"[Downloader] Downloading {url} -> {output_path}")
    headers = {"User-Agent": "DeepGuard-AI/1.0 (Google Colab)"}
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(output_path, "wb") as out_file:
        total_size = int(resp.headers.get("Content-Length", -1))
        downloaded = 0
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            out_file.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                percent = min(100.0, (downloaded / total_size) * 100.0)
                sys.stdout.write(f"\r  Downloaded: {downloaded / (1024*1024):.1f}/{total_size / (1024*1024):.1f} MB ({percent:.1f}%)")
                sys.stdout.flush()
    print("\n[Downloader] Download completed successfully.")
    return output_path

def extract_zip(zip_path, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    print(f"[Downloader] Extracting {zip_path} -> {extract_dir} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print("[Downloader] Extraction completed.")
    return extract_dir

def download_openrl_deepfake(target_dir="/content/data"):
    os.makedirs(target_dir, exist_ok=True)
    real_dir = os.path.join(target_dir, "real")
    fake_dir = os.path.join(target_dir, "fake")

    if not os.path.exists(real_dir):
        zip_path = os.path.join(target_dir, "wiki.zip")
        download_file_with_progress("https://huggingface.co/datasets/OpenRL/DeepFakeFace/resolve/main/wiki.zip", zip_path)
        extract_zip(zip_path, real_dir)
        try:
            os.remove(zip_path)
        except OSError:
            pass

    if not os.path.exists(fake_dir):
        zip_path = os.path.join(target_dir, "insight.zip")
        download_file_with_progress("https://huggingface.co/datasets/OpenRL/DeepFakeFace/resolve/main/insight.zip", zip_path)
        extract_zip(zip_path, fake_dir)
        try:
            os.remove(zip_path)
        except OSError:
            pass

    return real_dir, fake_dir

def build_manifest_from_folders(real_dir, fake_dir, manifest_path="/content/data/manifest.csv", max_per_class=None, val_ratio=0.15, test_ratio=0.15, seed=42):
    records = []
    real_files = glob.glob(os.path.join(real_dir, "**", "*.[jJ][pP]*[gG]"), recursive=True) + glob.glob(os.path.join(real_dir, "**", "*.png"), recursive=True)
    real_files.sort()
    if max_per_class:
        real_files = real_files[:max_per_class]

    for p in real_files:
        parent_name = os.path.basename(os.path.dirname(p))
        records.append({
            "image_path": p,
            "label": 0,
            "class_name": "REAL",
            "source_id": parent_name if parent_name and parent_name != "real" else os.path.splitext(os.path.basename(p))[0][:8]
        })

    fake_files = glob.glob(os.path.join(fake_dir, "**", "*.[jJ][pP]*[gG]"), recursive=True) + glob.glob(os.path.join(fake_dir, "**", "*.png"), recursive=True)
    fake_files.sort()
    if max_per_class:
        fake_files = fake_files[:max_per_class]

    for p in fake_files:
        parent_name = os.path.basename(os.path.dirname(p))
        records.append({
            "image_path": p,
            "label": 1,
            "class_name": "FAKE",
            "source_id": parent_name if parent_name and parent_name != "fake" else os.path.splitext(os.path.basename(p))[0][:8]
        })

    df = pd.DataFrame(records)
    print(f"[Manifest] Found {len(real_files)} REAL and {len(fake_files)} FAKE face crops.")

    unique_sources = df["source_id"].unique()
    np.random.seed(seed)
    np.random.shuffle(unique_sources)

    n_val = int(len(unique_sources) * val_ratio)
    n_test = int(len(unique_sources) * test_ratio)

    val_sources = set(unique_sources[:n_val])
    test_sources = set(unique_sources[n_val:n_val + n_test])

    def assign_split(s_id):
        if s_id in val_sources:
            return "val"
        elif s_id in test_sources:
            return "test"
        return "train"

    df["split"] = df["source_id"].apply(assign_split)
    os.makedirs(os.path.dirname(os.path.abspath(manifest_path)), exist_ok=True)
    df.to_csv(manifest_path, index=False)
    print(f"[Manifest] Saved manifest to {manifest_path}")
    print(f"  Train: {(df['split']=='train').sum()} samples")
    print(f"  Val:   {(df['split']=='val').sum()} samples")
    print(f"  Test:  {(df['split']=='test').sum()} samples")
    return df

os.makedirs(DATA_DIR, exist_ok=True)
max_limit = None if MAX_SAMPLES_PER_CLASS <= 0 else MAX_SAMPLES_PER_CLASS

if DATASET_SOURCE == "HuggingFace_OpenRL_DeepFakeFace":
    print("Downloading OpenRL/DeepFakeFace directly via URL (Zero login required)...")
    real_dir, fake_dir = download_openrl_deepfake(DATA_DIR)
    manifest_df = build_manifest_from_folders(
        real_dir=real_dir,
        fake_dir=fake_dir,
        manifest_path=os.path.join(DATA_DIR, "manifest.csv"),
        max_per_class=max_limit,
        val_ratio=0.15,
        test_ratio=0.15
    )
elif DATASET_SOURCE == "Kaggle_140k_Faces":
    !pip install -q opendatasets
    import opendatasets as od
    od.download("https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces", data_dir=DATA_DIR)
    real_dir = os.path.join(DATA_DIR, "140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real")
    fake_dir = os.path.join(DATA_DIR, "140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake")
    manifest_df = build_manifest_from_folders(real_dir, fake_dir, os.path.join(DATA_DIR, "manifest.csv"), max_per_class=max_limit)

print(f"\n>>> Dataset loaded and indexed: {len(manifest_df)} total samples.")

---
## Step 3: PyTorch DataLoaders & Preprocessing
Data transforms apply RGB normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`), horizontal flips, and color jitter to ensure robust artifact detection.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

# Select architecture to determine input resolution
ARCHITECTURE = "efficientnet_b0" #@param ["efficientnet_b0", "xception"]
INPUT_SIZE = 299 if ARCHITECTURE == "xception" else 224
BATCH_SIZE = 32 #@param {type:"integer"}

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class DeepfakeFaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])

train_df = manifest_df[manifest_df['split'] == 'train']
val_df = manifest_df[manifest_df['split'] == 'val']
test_df = manifest_df[manifest_df['split'] == 'test']

train_loader = DataLoader(DeepfakeFaceDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(DeepfakeFaceDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(DeepfakeFaceDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders created -> Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')


---
## Step 4: Model Architecture Initialization
We load the chosen pretrained backbone (`efficientnet_b0` or `xception`) and replace its classifier head with a binary classification head (`REAL`=0, `FAKE`=1).

In [ ]:
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Computing Device:', device)

def build_detector(arch_name):
    if arch_name == 'efficientnet_b0':
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_features = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, 2)
        )
        return base
    elif arch_name == 'xception':
        import timm
        try:
            return timm.create_model('xception', pretrained=True, num_classes=2)
        except Exception:
            return timm.create_model('legacy_xception', pretrained=True, num_classes=2)
    else:
        raise ValueError(f'Unknown architecture: {arch_name}')

model = build_detector(ARCHITECTURE).to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'>>> Model {ARCHITECTURE} initialized with {trainable_params:,} trainable parameters on {device}.')


---
## Step 5: Training Loop with AMP, AdamW & Early Stopping
Runs training with Automatic Mixed Precision (`torch.amp`), Cosine Annealing, and saves the best model based on validation loss.

In [ ]:
import time
from sklearn.metrics import accuracy_score, roc_auc_score

EPOCHS = 10 #@param {type:"integer"}
LR = 1e-4 #@param {type:"number"}
PATIENCE = 4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

print(f'>>> Starting training for {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    start_t = time.time()
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
    
    epoch_train_loss = running_loss / len(train_df)
    scheduler.step()
    
    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_preds, val_targets, val_probs = [], [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            val_probs.extend(probs.cpu().numpy())
            val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
    
    epoch_val_loss = val_loss / len(val_df)
    epoch_acc = accuracy_score(val_targets, val_preds)
    try:
        epoch_auc = roc_auc_score(val_targets, val_probs)
    except Exception:
        epoch_auc = 0.5
    
    elapsed = time.time() - start_t
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_acc)
    history['val_auc'].append(epoch_auc)
    
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_acc*100:.2f}% | Val AUC: {epoch_auc:.4f}')
    
    # Checkpoint logic
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_weights = model.state_dict().copy()
        patience_counter = 0
        print('  [*] Best validation loss improved. Checkpoint saved.')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping triggered after {epoch} epochs.')
            break

# Restore best model
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
print('>>> Training completed successfully!')


---
## Step 6: Test Set Evaluation & Metrics Visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import numpy as np

model.eval()
test_preds, test_targets, test_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        test_targets.extend(labels.numpy())

print('=== TEST SET CLASSIFICATION REPORT ===')
print(classification_report(test_targets, test_preds, target_names=['REAL', 'FAKE']))

cm = confusion_matrix(test_targets, test_preds)
print('Confusion Matrix:')
print(cm)

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss Curve
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC Curve
fpr, tpr, _ = roc_curve(test_targets, test_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='crimson', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Test Set ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Step 7: Export Checkpoint & 1-Click Browser Download
This cell packs your model into the exact DeepGuard AI format:
- `best_model.pth` (State dict with weights)
- `model_config.json` (Preprocessing & architecture configuration)

Running this automatically triggers your browser's download prompt!
When downloaded, simply move `best_model.pth` into your local project:
`07_DeepGuard_AI/backend/artifacts/models/<model_name>/best_model.pth`

In [ ]:
import json
from google.colab import files

EXPORT_DIR = f'/content/exported_models/{ARCHITECTURE}'
os.makedirs(EXPORT_DIR, exist_ok=True)

model_config = {
    'model_name': ARCHITECTURE,
    'display_name': 'EfficientNet-B0 Deepfake Detector' if ARCHITECTURE == 'efficientnet_b0' else 'XceptionNet Deepfake Detector',
    'architecture': ARCHITECTURE,
    'num_classes': 2,
    'classes': ['REAL', 'FAKE'],
    'positive_class': 'FAKE',
    'version': '1.0.0',
    'input_size': [INPUT_SIZE, INPUT_SIZE],
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
    'decision_threshold': 0.50,
    'preprocessing': {
        'face_crop_margin': 0.25,
        'interpolation': 'bilinear',
        'color_space': 'RGB'
    }
}

# Save config
config_path = os.path.join(EXPORT_DIR, 'model_config.json')
with open(config_path, 'w') as f:
    json.dump(model_config, f, indent=2)

# Save weights
checkpoint_path = os.path.join(EXPORT_DIR, 'best_model.pth')
torch.save({
    'state_dict': model.state_dict(),
    'config': model_config
}, checkpoint_path)

print(f'>>> Model saved to: {checkpoint_path}')
print('>>> Triggering browser download now...')
files.download(checkpoint_path)
files.download(config_path)
